# Box threat — Baseline + Exhaustive Feature Search

Naive baseline (`from_Box threat` only) vs all 127 combinations of delta TQ.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from itertools import combinations
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Style
BG = '#FAFAFA'; GRID = '#EDEDED'; AXIS = '#D5D5D5'
TEXT = '#2D2D2D'; SUBTEXT = '#737373'
C_BASELINE = '#BF5B3F'; C_TACTICAL = '#2E74B5'
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG,
    'axes.edgecolor': AXIS, 'axes.labelcolor': SUBTEXT,
    'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.titlepad': 16,
    'axes.labelsize': 9.5, 'axes.grid': False,
    'text.color': TEXT, 'xtick.color': SUBTEXT, 'ytick.color': SUBTEXT,
    'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5,
    'font.family': 'sans-serif', 'figure.dpi': 140,
    'axes.spines.top': False, 'axes.spines.right': False,
})

In [2]:
DATA_DIR = "../../../../thesis_data/processed_data/thesis_model_dataset/active/"
df = pd.read_parquet(DATA_DIR + "within_league_transfers_v5.parquet")
mf = df[df["from_position"] == "Midfielder"].copy()

QUALITY = 'Box threat'
mf["delta_Q"] = mf["to_" + QUALITY] - mf["from_" + QUALITY]

TEAM_QUALITIES = ["ATTACK", "ATTACKING_TRANSITION", "CHANCE_CREATION",
                  "DEFENCE", "DEFENSIVE_TRANSITION", "OUTCOME", "PENETRATION"]

for tq in TEAM_QUALITIES:
    mf[f"delta_tq_{tq}"] = mf[f"to_q_{tq}"] - mf[f"from_q_proj_{tq}"]

delta_tq = [f"delta_tq_{tq}" for tq in TEAM_QUALITIES]
mf = mf.dropna(subset=["from_" + QUALITY, "delta_Q"] + delta_tq)

train, test = train_test_split(mf, test_size=0.2, random_state=42)
y_train = train["delta_Q"]
y_test  = test["delta_Q"]
print(f"Quality: {QUALITY}")
print(f"Train: {len(train):,}  |  Test: {len(test):,}")

Quality: Box threat
Train: 3,932  |  Test: 984


---
## Naive Baseline

In [3]:
X_tr = sm.add_constant(train[["from_" + QUALITY]])
X_te = sm.add_constant(test[["from_" + QUALITY]])
naive = sm.OLS(y_train, X_tr).fit()
naive_pred = naive.predict(X_te)
naive_r2 = r2_score(y_test, naive_pred)
naive_mae = mean_absolute_error(y_test, naive_pred)
print(f"Naive — R²: {naive_r2:.4f}  |  MAE: {naive_mae:.4f}")

Naive — R²: 0.1535  |  MAE: 0.3845


---
## Exhaustive Search (127 combinations)

In [4]:
results = []

for k in range(1, len(delta_tq) + 1):
    for combo in combinations(delta_tq, k):
        feat = ["from_" + QUALITY] + list(combo)
        X_tr = sm.add_constant(train[feat])
        X_te = sm.add_constant(test[feat])
        model = sm.OLS(y_train, X_tr).fit()
        pred = model.predict(X_te)
        results.append({
            "n_deltas": k,
            "deltas": ", ".join(c.replace("delta_tq_", "") for c in combo),
            "R2_test": r2_score(y_test, pred),
            "MAE_test": mean_absolute_error(y_test, pred),
            "R2_adj_train": model.rsquared_adj,
            "BIC": model.bic,
        })

results_df = pd.DataFrame(results).sort_values('R2_test', ascending=False)
print(f"Total combinations: {len(results_df)}")

Total combinations: 127


---
## Top 10 by Test R²

In [5]:
print(f"Naive baseline R²: {naive_r2:.4f}")
print()
print(results_df.head(10).to_string(index=False, float_format="{:.4f}".format))

Naive baseline R²: 0.1535

 n_deltas                                                                            deltas  R2_test  MAE_test  R2_adj_train       BIC
        4                                             ATTACK, DEFENCE, OUTCOME, PENETRATION   0.2000    0.3748        0.1835 6013.2650
        5                       ATTACK, DEFENCE, DEFENSIVE_TRANSITION, OUTCOME, PENETRATION   0.1999    0.3747        0.1833 6021.5167
        3                                                          ATTACK, DEFENCE, OUTCOME   0.1994    0.3751        0.1836 6005.6529
        4                                    ATTACK, DEFENCE, DEFENSIVE_TRANSITION, OUTCOME   0.1994    0.3751        0.1834 6013.9072
        5                       ATTACK, ATTACKING_TRANSITION, DEFENCE, OUTCOME, PENETRATION   0.1989    0.3746        0.1837 6019.7524
        6 ATTACK, ATTACKING_TRANSITION, DEFENCE, DEFENSIVE_TRANSITION, OUTCOME, PENETRATION   0.1988    0.3746        0.1835 6027.9791
        5                   

---
## Best per Number of Deltas

In [6]:
best_per_k = results_df.loc[results_df.groupby('n_deltas')['R2_test'].idxmax()]
best_per_k = best_per_k.sort_values('n_deltas')
print(f"Naive baseline R²: {naive_r2:.4f}")
print()
print(best_per_k.to_string(index=False, float_format="{:.4f}".format))

Naive baseline R²: 0.1535

 n_deltas                                                                                             deltas  R2_test  MAE_test  R2_adj_train       BIC
        1                                                                                            OUTCOME   0.1902    0.3775        0.1783 6016.6093
        2                                                                                   DEFENCE, OUTCOME   0.1969    0.3753        0.1820 6006.0851
        3                                                                           ATTACK, DEFENCE, OUTCOME   0.1994    0.3751        0.1836 6005.6529
        4                                                              ATTACK, DEFENCE, OUTCOME, PENETRATION   0.2000    0.3748        0.1835 6013.2650
        5                                        ATTACK, DEFENCE, DEFENSIVE_TRANSITION, OUTCOME, PENETRATION   0.1999    0.3747        0.1833 6021.5167
        6                  ATTACK, ATTACKING_TRANSITION, DEFE

In [7]:
best = results_df.iloc[0]
best_deltas_str = best['deltas']
best_r2 = best['R2_test']
best_mae = best['MAE_test']
print('Best combo: ' + best_deltas_str)
print('R2 test: {:.4f}  |  MAE: {:.4f}'.format(best_r2, best_mae))
print('R2 gain over naive: {:+.4f}'.format(best_r2 - naive_r2))
print()

best_deltas = ['delta_tq_' + d.strip() for d in best_deltas_str.split(',')]
best_feat = ['from_' + QUALITY] + best_deltas
X_tr = sm.add_constant(train[best_feat])
best_model = sm.OLS(y_train, X_tr).fit()
print(best_model.summary())

Best combo: ATTACK, DEFENCE, OUTCOME, PENETRATION
R2 test: 0.2000  |  MAE: 0.3748
R2 gain over naive: +0.0465

                            OLS Regression Results                            
Dep. Variable:                delta_Q   R-squared:                       0.185
Model:                            OLS   Adj. R-squared:                  0.184
Method:                 Least Squares   F-statistic:                     177.7
Date:                Fri, 20 Mar 2026   Prob (F-statistic):          5.97e-171
Time:                        15:19:33   Log-Likelihood:                -2981.8
No. Observations:                3932   AIC:                             5976.
Df Residuals:                    3926   BIC:                             6013.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------